In [1]:
"""
如何把高像素,低语义的像素网络,逐步转化为低分辨率,高语义的特征表示
"""
import math
import dnnlpy
import dnnlpy.nn as dnn
import torch
import torch.nn as nn
import torchinfo
from torch import Tensor

print("PyTorcn Version", torch.__version__)

PyTorcn Version 2.13.0+cpu


In [ ]:
"""
leNet 解决了什么问题
    - 卷积层使用局部连接和权重共享提取特征
    - 池化层逐步降低空间分辨率
    - 更深层次的通道表示更复杂的模式
    - 全连接层根据最终特征完成分类
    - 所有参数都通过反向传播 共同学习
"""

In [7]:
"""
用PyTorch 实现经典的LeNet
"""


class LetNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            dnn.Conv2d(1, 6, kernel_size=5),
            dnn.ReLU(),
            dnn.AvgPool2d(kernel_size=2),
            dnn.Conv2d(6, 16, kernel_size=5),
            dnn.Tanh(),
            dnn.AvgPool2d(kernel_size=2),
            dnn.Conv2d(16, 120, kernel_size=5),
            dnn.Tanh(),
        )
        self.classifier = nn.Sequential(
            dnn.Flatten(),
            dnn.Linear(120, 84),
            dnn.Tanh(),
            dnn.Linear(84, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


model = LetNet5(num_classes=10)
x = torch.randn(8, 1, 32, 32)
logits = model(x)

print("model:", model, end='\n\n')
print("Input shape:", x.shape)
print("Output shape:", logits.shape)

loss_fn = nn.CrossEntropyLoss()

print("loss fn:", loss_fn)

summary = torchinfo.summary(model, input_size=(8, 1, 32, 32))
print(summary)

model: LetNet5(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Tanh()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
    (7): Tanh()
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=120, out_features=84, bias=True)
    (2): Tanh()
    (3): Linear(in_features=84, out_features=10, bias=True)
  )
)

Input shape: torch.Size([8, 1, 32, 32])
Output shape: torch.Size([8, 10])
loss fn: CrossEntropyLoss()
Layer (type:depth-idx)                   Output Shape              Param #
LetNet5                                  [8, 10]                   --
├─Sequential: 1-1                        [8, 120, 1, 1]            --
│    └─Conv2d: 2-1                       [8, 6, 28, 28]            156
│    └─ReLU: 2-2   

In [9]:
"""
    原始写法和现代写法有什么不同
        - 用RelU()代替Tanh
        - 使用max_pooling 或者stride convolution 代替average pooling
        - 使用padding,使卷积神的空间尺寸更容易控制
        - 使用adaptive pooling ,避免分类头依赖固定的分辨率
        - 根据需要加入normalization和dropout
"""


class ModernLeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            dnn.Conv2d(1, 16, kernel_size=3, padding=1),
            dnn.ReLU(),
            dnn.MaxPool2d(kernel_size=2),
            dnn.Conv2d(16, 32, kernel_size=3, padding=1),
            dnn.ReLU(),
            dnn.MaxPool2d(kernel_size=2),
        )

        self.pool = dnn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.classifier = dnn.Linear(32, num_classes)

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x


model = ModernLeNet5(num_classes=10)
summary = torchinfo.summary(model, input_size=(8, 1, 32, 32))
print(summary)


Layer (type:depth-idx)                   Output Shape              Param #
ModernLeNet5                             [8, 10]                   --
├─Sequential: 1-1                        [8, 32, 8, 8]             --
│    └─Conv2d: 2-1                       [8, 16, 32, 32]           160
│    └─ReLU: 2-2                         [8, 16, 32, 32]           --
│    └─MaxPool2d: 2-3                    [8, 16, 16, 16]           --
│    └─Conv2d: 2-4                       [8, 32, 16, 16]           4,640
│    └─ReLU: 2-5                         [8, 32, 16, 16]           --
│    └─MaxPool2d: 2-6                    [8, 32, 8, 8]             --
├─AdaptiveAvgPool2d: 1-2                 [8, 32, 1, 1]             --
├─Flatten: 1-3                           [8, 32]                   --
├─Linear: 1-4                            [8, 10]                   330
Total params: 5,130
Trainable params: 5,130
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 10.82
Input size (MB): 0.03
Forward/backward pa

In [15]:
"""
最后一层卷积和全连接层的关系
"""
x = torch.randn(3, 16, 5, 5)
conv = nn.Conv2d(16, 120, kernel_size=5)
linear = nn.Linear(16 * 5 * 5, 120)

with torch.no_grad():
    linear.weight.copy_(conv.weight.reshape(120,-1))
    linear.bias.copy_(conv.bias)

conv_output = conv(x).flatten(start_dim=1)
linear_output= linear(x.flatten(start_dim=1))

max_diff =(conv_output - linear_output).abs().max()
print("max_diff:", max_diff.item())


max_diff: 1.3113021850585938e-06


In [ ]:
"""
leNet 留下了什么
    先用卷积层逐层提取空间特征,再把最终表示的结果给分类器
"""